# 🏏 IPL Win & Score Prediction — Google Colab Training

This notebook trains the full IPL prediction pipeline end-to-end on Google Colab.

**What it does:**
1. Installs all dependencies
2. Downloads the latest IPL ball-by-ball data from Cricsheet
3. Clones your project code (or lets you upload it)
4. Runs the preprocessing pipeline → generates `ipl_features.csv`
5. Trains Score models: **HGB · XGBoost · CatBoost · RandomForest · PyTorch DL**
6. Trains Win models: same families + isotonic calibration
7. Compares all models and promotes the best ones
8. Saves trained models to Google Drive / download

**Recommended runtime:** `Runtime → Change runtime type → T4 GPU`

---

## ⚙️ Step 0 — Configuration
Fill in the settings below before running anything else.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIG — edit these before running
# ─────────────────────────────────────────────────────────────────────────────

# Your GitHub repo URL (leave blank to upload a zip instead)
GITHUB_REPO_URL = ""  # e.g. "https://github.com/soulrahulrk/ipl-prediction.git"

# Branch name (usually main or master)
GITHUB_BRANCH = "main"

# Mount Google Drive to save trained models there?
USE_GOOGLE_DRIVE = True

# If using Drive, where to save results inside your Drive
DRIVE_SAVE_PATH = "/content/drive/MyDrive/ipl_models"

# Skip PyTorch DL training? (faster if True, but DL is best on GPU)
SKIP_DL = False

# Skip slow models (RF) for a quick test run?
QUICK_MODE = False

# Re-download + reprocess data even if ipl_features.csv already exists?
FORCE_REPROCESS = False

print("Config loaded.")
print(f"  GitHub repo   : {GITHUB_REPO_URL or '(upload manually)'}")
print(f"  Google Drive  : {USE_GOOGLE_DRIVE}")
print(f"  Skip DL       : {SKIP_DL}")
print(f"  Quick mode    : {QUICK_MODE}")
print(f"  Force reprocess: {FORCE_REPROCESS}")

Config loaded.
  GitHub repo   : (upload manually)
  Google Drive  : True
  Skip DL       : False
  Quick mode    : False
  Force reprocess: False


## 📦 Step 1 — Install Dependencies

In [ ]:
%%time
# Core ML & data science packages for training
# (no Flask / web / DB deps needed for training)
!pip install -q \
    "numpy>=1.26,<3" \
    "pandas>=2.2,<3" \
    "scikit-learn>=1.7,<1.8" \
    "joblib>=1.4,<2" \
    "xgboost>=2.1,<3" \
    "catboost>=1.2,<2" \
    "matplotlib>=3.9,<4" \
    "tqdm"

print("\n✅ Packages installed.")

In [ ]:
# Verify GPU availability for XGBoost / CatBoost / PyTorch
import torch

GPU_AVAILABLE = torch.cuda.is_available()
if GPU_AVAILABLE:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA version : {torch.version.cuda}")
    print(f"   PyTorch      : {torch.__version__}")
else:
    print("⚠️  No GPU detected — training will use CPU (slower).")
    print("   Go to Runtime → Change runtime type → T4 GPU for faster training.")

## 💾 Step 2 — Mount Google Drive (optional)
Skip this cell if `USE_GOOGLE_DRIVE = False`.

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
    print(f"✅ Google Drive mounted. Models will be saved to: {DRIVE_SAVE_PATH}")
else:
    print("Google Drive not mounted. Models will only be available for download.")

## 📂 Step 3 — Get Project Code

**Option A** (recommended): Set `GITHUB_REPO_URL` above and run the next cell to clone your repo.

**Option B**: Upload your project as a zip file — run the cell after.

**Option C**: If you have the project on Google Drive, copy it.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_DIR = Path("/content/ipl_prediction")

if GITHUB_REPO_URL:
    # ── Option A: Clone from GitHub ────────────────────────────────────────
    if PROJECT_DIR.exists():
        print(f"Directory {PROJECT_DIR} already exists, pulling latest changes...")
        !cd /content/ipl_prediction && git pull
    else:
        print(f"Cloning {GITHUB_REPO_URL} ...")
        !git clone --depth 1 --branch {GITHUB_BRANCH} {GITHUB_REPO_URL} /content/ipl_prediction
    print("\n✅ Project code ready.")
else:
    print("⚠️  GITHUB_REPO_URL is empty.")
    print("Run the next cell to upload your project zip instead.")

In [ ]:
# ── Option B: Upload project zip ──────────────────────────────────────────────
# Only run this cell if you did NOT set GITHUB_REPO_URL above.
# 1. Zip your entire ipl prediction project folder on your machine
# 2. Run this cell and upload the zip when prompted

if not GITHUB_REPO_URL:
    from google.colab import files
    print("Upload your project zip file (e.g. ipl_prediction.zip):")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -q "{zip_name}" -d /content/
    # Try to find the project root
    import glob
    candidates = glob.glob("/content/*/ipl_predictor") + glob.glob("/content/ipl_predictor")
    if candidates:
        PROJECT_DIR = Path(candidates[0]).parent
        print(f"✅ Project found at: {PROJECT_DIR}")
    else:
        print("⚠️  Could not auto-detect project root. Set PROJECT_DIR manually.")
        # PROJECT_DIR = Path("/content/your-folder-name")  # ← uncomment and edit if needed
else:
    print("Using GitHub clone — skipping upload.")

In [ ]:
# Add project root to Python path so we can import ipl_predictor
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Quick sanity check
try:
    import ipl_predictor
    from ipl_predictor import (
        CATEGORICAL_FEATURES, NUMERIC_FEATURES, ACTIVE_IPL_TEAMS_2026,
        PROCESSED_DIR, MODELS_DIR, DATA_DIR
    )
    print("✅ ipl_predictor package imported successfully.")
    print(f"   Categorical features : {len(CATEGORICAL_FEATURES)}")
    print(f"   Numeric features     : {len(NUMERIC_FEATURES)}")
    print(f"   Active teams         : {len(ACTIVE_IPL_TEAMS_2026)}")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Check that PROJECT_DIR points to the folder containing ipl_predictor/")

## 🏏 Step 4 — Download IPL Data from Cricsheet

This downloads the complete IPL ball-by-ball dataset (~30 MB zip, ~900+ matches) directly from [cricsheet.org](https://cricsheet.org).

If you already have the data on Google Drive, copy it instead.

In [ ]:
%%time
import io, urllib.request, zipfile
from pathlib import Path

RAW_DATA_DIR = PROJECT_DIR / "data" / "ipl_csv2"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

existing_csvs = list(RAW_DATA_DIR.glob("*.csv"))

if existing_csvs and not FORCE_REPROCESS:
    print(f"✅ Found {len(existing_csvs)} CSV files in {RAW_DATA_DIR}")
    print("   Skipping download. Set FORCE_REPROCESS=True to re-download.")
else:
    CRICSHEET_URL = "https://cricsheet.org/downloads/ipl_csv2.zip"
    print(f"Downloading IPL CSV2 data from {CRICSHEET_URL} ...")
    print("(This may take 1-3 minutes depending on Cricsheet server load)")

    req = urllib.request.Request(
        CRICSHEET_URL,
        headers={
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36",
            "Accept": "application/zip, */*",
        }
    )

    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(req, timeout=180) as resp:
                if resp.status != 200:
                    raise RuntimeError(f"HTTP {resp.status}")
                payload = resp.read()
            print(f"   Downloaded {len(payload)/1e6:.1f} MB")
            break
        except Exception as exc:
            print(f"   Attempt {attempt} failed: {exc}")
            if attempt < 3:
                import time; time.sleep(5 * attempt)
    else:
        raise RuntimeError("Failed to download after 3 attempts. Check Cricsheet.org manually.")

    print("Extracting archive...")
    with zipfile.ZipFile(io.BytesIO(payload)) as zf:
        zf.extractall(RAW_DATA_DIR)

    csvs = list(RAW_DATA_DIR.glob("*.csv"))
    print(f"✅ Extracted {len(csvs)} files to {RAW_DATA_DIR}")
    # Count match files (not _info files)
    match_files = [f for f in csvs if not f.stem.endswith("_info")]
    print(f"   Match files  : {len(match_files)}")
    print(f"   Info files   : {len(csvs) - len(match_files)}")

In [ ]:
# ── Alternative: Copy from Google Drive (if you already have the data there) ──
# Uncomment and edit the path below if you want to use Drive data instead:

# DRIVE_DATA_PATH = "/content/drive/MyDrive/ipl_data/ipl_csv2"
# import shutil
# if Path(DRIVE_DATA_PATH).exists():
#     shutil.copytree(DRIVE_DATA_PATH, str(RAW_DATA_DIR), dirs_exist_ok=True)
#     print(f"Copied data from {DRIVE_DATA_PATH}")
# else:
#     print(f"Drive path not found: {DRIVE_DATA_PATH}")

print("(Drive copy cell — uncomment if needed)")

## 🔧 Step 5 — Preprocess Data

Converts raw Cricsheet ball-by-ball CSV files into a single engineered features CSV.

**Output**: `data/processed/ipl_features.csv` (~140 MB, millions of rows)

**This step takes ~10–20 minutes** on Colab CPU. It only runs if the file doesn't exist yet.

In [ ]:
%%time
import os
from pathlib import Path

FEATURES_PATH = PROJECT_DIR / "data" / "processed" / "ipl_features.csv"

if FEATURES_PATH.exists() and not FORCE_REPROCESS:
    size_mb = FEATURES_PATH.stat().st_size / 1e6
    print(f"✅ ipl_features.csv already exists ({size_mb:.1f} MB).")
    print("   Skipping preprocessing. Set FORCE_REPROCESS=True to redo it.")
else:
    print("Running preprocessing pipeline...")
    print("(Expected time: 10–20 minutes on Colab CPU)")
    print()
    !cd {PROJECT_DIR} && python scripts/preprocess_ipl.py
    if FEATURES_PATH.exists():
        size_mb = FEATURES_PATH.stat().st_size / 1e6
        print(f"\n✅ Preprocessing complete. ipl_features.csv: {size_mb:.1f} MB")
    else:
        print("\n❌ ipl_features.csv not found after preprocessing. Check errors above.")

## 📊 Step 6 — Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.style.use("seaborn-v0_8-whitegrid")

print("Loading ipl_features.csv ...")
df = pd.read_csv(FEATURES_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"Seasons: {sorted(df['season'].dropna().unique())}")
df.head(3)

In [ ]:
# Season distribution
from ipl_predictor import season_to_year

df["season_int"] = df["season"].apply(season_to_year)
season_counts = df.groupby("season_int")["match_id"].nunique().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

season_counts.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Matches per Season")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Matches")
axes[0].tick_params(axis="x", rotation=45)

df.groupby("batting_team")["match_id"].nunique().sort_values(ascending=False).head(12).plot(
    kind="bar", ax=axes[1], color="coral"
)
axes[1].set_title("Innings Appearances per Team (top 12)")
axes[1].set_xlabel("Team")
axes[1].set_ylabel("Innings")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Total rows: {len(df):,}  |  Unique matches: {df['match_id'].nunique()}")

In [ ]:
# Score & win distributions
valid_scores = df[df["total_runs"].notna() & (df["innings"] == 1)]["total_runs"]
valid_wins   = df[df["win"].notna()]["win"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(valid_scores, bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("1st Innings Final Score Distribution")
axes[0].set_xlabel("Runs")
axes[0].set_ylabel("Frequency")

win_counts = valid_wins.value_counts()
axes[1].bar(["Batting team wins (1)", "Batting team loses (0)"],
            [win_counts.get(1.0, 0), win_counts.get(0.0, 0)],
            color=["steelblue", "coral"])
axes[1].set_title("Win Label Distribution")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Score (1st innings): mean={valid_scores.mean():.1f}  median={valid_scores.median():.1f}  std={valid_scores.std():.1f}")
print(f"Win label balance  : {win_counts.to_dict()}")

In [ ]:
# Venue averages
venue_avg = (
    df[df["innings"] == 1]
    .groupby("venue")["total_runs"]
    .agg(["mean", "count"])
    .query("count >= 10")
    .sort_values("mean", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 5))
venue_avg["mean"].plot(kind="barh", ax=ax, color="teal")
ax.set_title("Average 1st Innings Score by Venue (top 15, min 10 matches)")
ax.set_xlabel("Average Score")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 🤖 Step 7 — Train All Models

Runs the full multi-model training pipeline:
- **Train split**: all seasons except last 2  
- **Validation**: season `[-2]` (early stopping / model selection)  
- **Test**: season `[-1]` (hold-out evaluation)

Best score + win models are promoted to `score_model.pkl` / `win_model.pkl`.

In [ ]:
%%time
skip_dl_flag  = "--skip-dl" if SKIP_DL  else ""
quick_flag    = "--quick"   if QUICK_MODE else ""

cmd = f"cd {PROJECT_DIR} && python scripts/train_all_models.py {skip_dl_flag} {quick_flag}"
print(f"Running: {cmd.strip()}")
print("="*60)
!{cmd}

## 📈 Step 8 — Inspect Results

In [ ]:
import json

report_path = PROJECT_DIR / "models" / "all_models_report.json"
if not report_path.exists():
    print("Report not found — training may not have completed.")
else:
    with open(report_path) as f:
        report = json.load(f)

    print(f"Trained at  : {report.get('trained_at')}")
    print(f"Test season : {report.get('test_season')}")
    print(f"Total time  : {report.get('total_train_secs')}s")
    print(f"Promoted    : {report.get('promoted')}")

In [ ]:
# Score model comparison table
if report_path.exists():
    score_df = pd.DataFrame(report["score_models"]).sort_values("test_rmse")
    best_score = report["promoted"].get("score", "")

    def highlight_best(row):
        return ["background-color: #d4edda" if row["model"] == best_score else "" for _ in row]

    print("=== SCORE MODEL COMPARISON (lower RMSE = better) ===")
    display(
        score_df[["model", "test_mae", "test_rmse", "train_secs"]]
        .style.apply(highlight_best, axis=1)
        .format({"test_mae": "{:.3f}", "test_rmse": "{:.3f}", "train_secs": "{:.0f}s"})
        .set_caption("✅ Green = promoted production model")
    )

In [ ]:
# Win model comparison table
if report_path.exists():
    win_df = pd.DataFrame(report["win_models"]).sort_values("test_log_loss")
    best_win = report["promoted"].get("win", "")

    def highlight_best_win(row):
        return ["background-color: #d4edda" if row["model"] == best_win else "" for _ in row]

    print("=== WIN MODEL COMPARISON (lower LogLoss = better) ===")
    display(
        win_df[["model", "test_accuracy", "test_log_loss", "test_brier", "train_secs"]]
        .style.apply(highlight_best_win, axis=1)
        .format({"test_accuracy": "{:.4f}", "test_log_loss": "{:.4f}",
                 "test_brier": "{:.4f}", "train_secs": "{:.0f}s"})
        .set_caption("✅ Green = promoted production model")
    )

In [ ]:
# Visual comparison
if report_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Score: RMSE bar chart
    sd = score_df.set_index("model")
    colors = ["#2ecc71" if m == best_score else "#3498db" for m in sd.index]
    sd["test_rmse"].plot(kind="barh", ax=axes[0], color=colors)
    axes[0].set_title("Score Model — Test RMSE (lower is better)")
    axes[0].set_xlabel("RMSE (runs)")
    axes[0].invert_yaxis()
    axes[0].axvline(sd["test_rmse"].min(), color="green", linestyle="--", alpha=0.5)

    # Win: Log-loss bar chart
    wd = win_df.set_index("model")
    colors_w = ["#2ecc71" if m == best_win else "#e74c3c" for m in wd.index]
    wd["test_log_loss"].plot(kind="barh", ax=axes[1], color=colors_w)
    axes[1].set_title("Win Model — Test Log-Loss (lower is better)")
    axes[1].set_xlabel("Log-Loss")
    axes[1].invert_yaxis()
    axes[1].axvline(wd["test_log_loss"].min(), color="green", linestyle="--", alpha=0.5)

    plt.suptitle("Model Comparison — Green = Promoted", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

## ✅ Step 9 — Verify Promoted Models

Quick sanity-check: load the promoted models and run a test prediction.

In [ ]:
import joblib
from pathlib import Path

MODELS_DIR = PROJECT_DIR / "models"
score_model_path = MODELS_DIR / "score_model.pkl"
win_model_path   = MODELS_DIR / "win_model.pkl"

if score_model_path.exists():
    score_model = joblib.load(score_model_path)
    print(f"✅ score_model.pkl loaded  ({score_model_path.stat().st_size/1e6:.1f} MB)")
    print(f"   Type: {type(score_model).__name__}")
else:
    print("❌ score_model.pkl not found")

if win_model_path.exists():
    win_model = joblib.load(win_model_path)
    print(f"✅ win_model.pkl loaded    ({win_model_path.stat().st_size/1e6:.1f} MB)")
    print(f"   Type: {type(win_model).__name__}")
else:
    print("❌ win_model.pkl not found")

In [ ]:
# Quick end-to-end prediction test using the full prediction pipeline
try:
    from ipl_predictor import load_models, load_support_tables, predict_match_state

    # Override model dir if not standard
    models = load_models()
    support = load_support_tables()

    test_input = {
        "season":        "2024",
        "venue":         "Wankhede Stadium",
        "batting_team":  "Mumbai Indians",
        "bowling_team":  "Royal Challengers Bengaluru",
        "innings":       1,
        "runs":          85,
        "wickets":       3,
        "overs":         "12.0",
        "toss_winner":   "Mumbai Indians",
        "toss_decision": "bat",
        "striker":       "Unknown",
        "bowler":        "Unknown",
        "runs_last_5":   28,
    }

    result = predict_match_state(test_input, models, support)
    print("✅ Test prediction succeeded:")
    for k, v in result.items():
        print(f"   {k:<30} {v}")
except Exception as e:
    print(f"⚠️  Prediction test failed (non-critical): {e}")
    print("   The models are still trained and saved correctly.")

## 💾 Step 10 — Save / Download Models

In [ ]:
import shutil, os
from pathlib import Path
from datetime import datetime

# Files to save
MODEL_FILES = [
    "score_model.pkl",
    "win_model.pkl",
    "score_model_hgb.pkl",
    "score_model_xgb2.pkl",
    "score_model_cat2.pkl",
    "win_model_hgb.pkl",
    "win_model_xgb2.pkl",
    "win_model_cat2.pkl",
    "all_models_report.json",
    "score_uncertainty.json",
]

if USE_GOOGLE_DRIVE:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    save_dir = Path(DRIVE_SAVE_PATH) / timestamp
    save_dir.mkdir(parents=True, exist_ok=True)

    copied = []
    for fname in MODEL_FILES:
        src = MODELS_DIR / fname
        if src.exists():
            shutil.copy(src, save_dir / fname)
            copied.append(fname)

    print(f"✅ Saved {len(copied)} files to Google Drive: {save_dir}")
    for f in copied:
        sz = (save_dir / f).stat().st_size / 1e6
        print(f"   {f:<35} {sz:.1f} MB")
else:
    print("Google Drive not mounted. Run the download cell below to get the files.")

In [ ]:
# Create a single zip of all model artifacts for easy download
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_path = Path(f"/content/ipl_models_{timestamp}.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in MODEL_FILES:
        src = MODELS_DIR / fname
        if src.exists():
            zf.write(src, fname)
    # Also include processed support tables
    processed_dir = PROJECT_DIR / "data" / "processed"
    for support_file in [
        "venue_stats.csv", "team_form_latest.csv", "team_venue_form_latest.csv",
        "matchup_form_latest.csv", "batter_form_latest.csv", "bowler_form_latest.csv",
        "active_teams_2026.csv", "team_player_pool_2026.csv",
    ]:
        src = processed_dir / support_file
        if src.exists():
            zf.write(src, f"processed/{support_file}")

print(f"✅ Created archive: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Download the zip to your local machine
from google.colab import files
print(f"Downloading {zip_path.name} ...")
files.download(str(zip_path))
print("Done! Check your browser's download folder.")

## 🏋️ (Optional) Step 11 — Train Pre-Match Models

Trains the simpler pre-match models that only need team names + venue (no live match state).

In [ ]:
%%time
!cd {PROJECT_DIR} && python scripts/train_pre_match.py

## 🔬 (Optional) Step 12 — Deep Dive: Feature Importance

In [ ]:
from ipl_predictor import CATEGORICAL_FEATURES, NUMERIC_FEATURES
import joblib

ALL_FEATS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Load the HGB score model (always trained, simple sklearn Pipeline)
hgb_path = MODELS_DIR / "score_model_hgb.pkl"
if not hgb_path.exists():
    print("score_model_hgb.pkl not found — run training first.")
else:
    hgb = joblib.load(hgb_path)
    # Extract feature importances from the HGB step
    from sklearn.pipeline import Pipeline
    if isinstance(hgb, Pipeline):
        booster = hgb.named_steps["m"]
        importances = booster.feature_importances_

        # The preprocessor reorders features (numeric first, then encoded cats)
        # For HGB with ordinal encoding, feature count = len(NUM_FEATS) + len(CAT_FEATS)
        n_num = len(NUMERIC_FEATURES)
        n_cat = len(CATEGORICAL_FEATURES)
        feat_names = NUMERIC_FEATURES + CATEGORICAL_FEATURES

        if len(importances) == len(feat_names):
            imp_df = pd.Series(importances, index=feat_names).sort_values(ascending=False)

            fig, ax = plt.subplots(figsize=(10, 8))
            colors = ["coral" if f in CATEGORICAL_FEATURES else "steelblue" for f in imp_df.index[:20]]
            imp_df.head(20).plot(kind="barh", ax=ax, color=colors)
            ax.set_title("HGB Score Model — Top 20 Feature Importances\n(blue=numeric, red=categorical)")
            ax.set_xlabel("Importance")
            ax.invert_yaxis()
            plt.tight_layout()
            plt.show()
        else:
            print(f"Feature count mismatch: {len(importances)} importances vs {len(feat_names)} names.")
            print("This happens with OHE encoding — showing raw importances.")
            pd.Series(importances).nlargest(20).plot(kind="bar", figsize=(12, 4), title="Top 20 Feature Importances")
            plt.tight_layout()
            plt.show()

In [ ]:
# CatBoost feature importance (if trained)
cat_path = MODELS_DIR / "score_model_cat2.pkl"
if cat_path.exists():
    cb_model = joblib.load(cat_path)
    from catboost import CatBoostRegressor
    if isinstance(cb_model, CatBoostRegressor):
        imp = pd.Series(
            cb_model.get_feature_importance(),
            index=cb_model.feature_names_
        ).sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(10, 8))
        colors = ["coral" if f in CATEGORICAL_FEATURES else "steelblue" for f in imp.index[:20]]
        imp.head(20).plot(kind="barh", ax=ax, color=colors)
        ax.set_title("CatBoost Score Model — Top 20 Feature Importances")
        ax.set_xlabel("Importance")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("CatBoost model not found (may have been skipped in quick mode).")

## 🔬 (Optional) Step 13 — Residual Analysis

In [ ]:
# Load test split and compute residuals for the promoted score model
from ipl_predictor import CATEGORICAL_FEATURES, NUMERIC_FEATURES, ACTIVE_IPL_TEAMS_2026, season_to_year
import joblib
import numpy as np

ALL_FEATS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Reload df if needed
if "df" not in dir() or "season_int" not in df.columns:
    df = pd.read_csv(FEATURES_PATH, low_memory=False)
    df["season_int"] = df["season"].apply(season_to_year)

mask = (
    df["batting_team"].isin(ACTIVE_IPL_TEAMS_2026)
    & df["bowling_team"].isin(ACTIVE_IPL_TEAMS_2026)
)
seasons = sorted(df.loc[mask, "season_int"].dropna().unique())
test_yr = int(seasons[-1])

test = df[mask & (df["season_int"] == test_yr) & (df["balls_left"] > 0) & df["total_runs"].notna()]
print(f"Test year: {test_yr}  |  Test rows: {len(test):,}")

score_model_path = MODELS_DIR / "score_model.pkl"
if score_model_path.exists():
    model = joblib.load(score_model_path)
    X_te = test[ALL_FEATS]

    # Handle dict-wrapped models (XGBoost, RF)
    if isinstance(model, dict):
        preds = model["model"].predict(model["pre"].transform(X_te))
    else:
        preds = model.predict(X_te)

    residuals = test["total_runs"].values - preds
    mae  = np.mean(np.abs(residuals))
    rmse = np.sqrt(np.mean(residuals**2))

    print(f"\nPromoted score model — Test {test_yr}:")
    print(f"  MAE  = {mae:.2f} runs")
    print(f"  RMSE = {rmse:.2f} runs")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].scatter(test["total_runs"], preds, alpha=0.1, s=1, color="steelblue")
    mn, mx = test["total_runs"].min(), test["total_runs"].max()
    axes[0].plot([mn, mx], [mn, mx], "r--", lw=1)
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predicted")
    axes[0].set_title("Actual vs Predicted")

    axes[1].hist(residuals, bins=60, color="steelblue", edgecolor="white")
    axes[1].axvline(0, color="red", linestyle="--")
    axes[1].set_xlabel("Residual (actual − predicted)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Residual Distribution  (MAE={mae:.1f})")

    axes[2].scatter(preds, residuals, alpha=0.1, s=1, color="coral")
    axes[2].axhline(0, color="blue", linestyle="--")
    axes[2].set_xlabel("Predicted")
    axes[2].set_ylabel("Residual")
    axes[2].set_title("Residuals vs Predicted")

    plt.tight_layout()
    plt.show()
else:
    print("score_model.pkl not found.")

In [ ]:
# Win probability calibration curve
from sklearn.calibration import CalibrationDisplay

win_model_path = MODELS_DIR / "win_model.pkl"
if win_model_path.exists():
    win_model = joblib.load(win_model_path)
    test_w = df[mask & (df["season_int"] == test_yr) & (df["balls_left"] > 0) & df["win"].notna()]
    X_te_w = test_w[ALL_FEATS]
    y_te_w = test_w["win"].astype(int)

    if isinstance(win_model, dict):
        probs = win_model["model"].predict_proba(win_model["pre"].transform(X_te_w))[:, 1]
    else:
        probs = win_model.predict_proba(X_te_w)[:, 1]

    from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
    acc   = accuracy_score(y_te_w, (probs >= 0.5).astype(int))
    ll    = log_loss(y_te_w, probs)
    brier = brier_score_loss(y_te_w, probs)

    fig, ax = plt.subplots(figsize=(6, 5))
    CalibrationDisplay.from_predictions(y_te_w, probs, n_bins=15, ax=ax, name="Promoted Win Model")
    ax.set_title(f"Win Probability Calibration\nAcc={acc:.4f}  LogLoss={ll:.4f}  Brier={brier:.4f}")
    plt.tight_layout()
    plt.show()
else:
    print("win_model.pkl not found.")

## 📋 Summary

After running all cells above you will have:

| File | Description |
|------|-------------|
| `models/score_model.pkl` | Best score prediction model (promoted) |
| `models/win_model.pkl` | Best win probability model (promoted) |
| `models/score_model_hgb.pkl` | HistGradientBoosting score model |
| `models/score_model_xgb2.pkl` | XGBoost score model |
| `models/score_model_cat2.pkl` | CatBoost score model |
| `models/win_model_hgb.pkl` | HGB win model (calibrated) |
| `models/win_model_xgb2.pkl` | XGBoost win model (calibrated) |
| `models/win_model_cat2.pkl` | CatBoost win model |
| `models/all_models_report.json` | Full comparison metrics |
| `models/score_uncertainty.json` | Residual uncertainty profile |

**Copy these back into your local `models/` folder to use them in the web app.**

---
*Trained on IPL data from [cricsheet.org](https://cricsheet.org)*